# Lab 02 · RAG: ground answers in retrieved knowledge

**Prepared by Mustafa Assaf** · Lecture 1: Ask, check, retrieve · ~15–20 min core work (estimate; setup and extras take longer)

## Start with a small example

First learn the mechanism with an everyday example. Then apply it to the software architecture exercise below.

## Setup
Run this cell once to load the course helpers, then run the steps below in order. Choose `PROVIDER` and save its key in Colab Secrets. DeepSeek uses `DEEPSEEK_API_KEY`; Gemini and FreeInference use their respective secrets. `client` stops after 100 calls; rerun setup to reset the count.

In [ ]:
from importlib.util import find_spec
from importlib.metadata import PackageNotFoundError, version
try:
    coursekit_version = version("agentic-ai-coursekit")
except PackageNotFoundError:
    coursekit_version = None
if coursekit_version != "0.5.1" or not all(find_spec(name) for name in ['coursekit', 'chromadb']):
    %pip install -q --upgrade "agentic-ai-coursekit[rag] @ https://github.com/mkassaf/agentic-ai-course/archive/a0b8f10a4139527a62c3a566286b9086f4aea7fe.zip"
    import importlib, sys
    for name in list(sys.modules):
        if name == "coursekit" or name.startswith("coursekit."):
            del sys.modules[name]
    importlib.invalidate_caches()
from coursekit import Client, PROVIDERS, secret
PROVIDER = "deepseek"  # or "gemini" or "freeinference"
settings = PROVIDERS[PROVIDER]
client = Client(model=settings["model"], family=settings["family"],
                base_url=settings["base_url"], api_key=secret(settings["secret_name"]),
                max_calls=100, max_tokens=256, extra_body=settings.get("extra_body"))

## Warm-up · Find a note before answering

We already know the museum note is relevant. The code picks it from a Python dictionary and puts it into the same prompt as the question. Look for the note under `Retrieved` and the opening time in the model’s reply.

In [ ]:
notes = {"museum": "The museum opens at 10:00.", "park": "The park opens at sunrise."}
question = "When does the museum open?"
retrieved = notes["museum"]  # The application chose the key for this example.
print("Retrieved:", retrieved)
answer = client.ask(f"Use this note: {retrieved}\nQuestion: {question}")
print("Answer:", answer)

**Next:** Next, let a search program choose which microservices policies to put in the model’s prompt.

## RAG: find the right microservices policy
The ShopEasy team has a library of policies about service databases, failures, and messages. Suppose you ask whether Reporting can read another service's database. The model needs the policy on data ownership to answer.

**Retrieval-augmented generation (RAG)** means searching for relevant material first, then including it in the prompt that asks the model to answer. The search step is called **retrieval**. The model’s answer is called **generation**.

You will ask the same three questions with all nine policies, then with only two search results per question. The library addresses two questions. The third asks about lunch so you can see what happens when the evidence cannot answer.

Chroma stores the searchable policies. A small local model turns text into numbers for the search. The first run downloads that small model. The answer calls still use the provider and key you set in setup.

**At a glance**

![Lab 02 at a glance: Policy index (embed + store) or Question (embed query), then Retrieve (top 2 policies), then Grounded answer (question + evidence), then Check citations (IDs + support).](https://mkassaf.github.io/agentic-ai-course/assets/diagrams/02.svg)

## Think about the example

API composition asks services for data and joins their responses. CQRS maintains a separate read model for queries, which can lag behind writes. A citation identifies a source; it does not prove the answer follows from it.

**Concepts:** Database per service; API composition vs CQRS; Idempotent consumers and transactional outbox.

Use the scenario above to consider these questions before running the code:

1. Why is CQRS a better fit for searchable order history than a large in-memory API join?
2. Does citing a valid source ID prove that the source supports an answer?

## The idea in context

Sending every policy increases prompt size as a library grows. RAG retrieves selected evidence; embeddings and nearest-neighbour scores do not guarantee relevance.

Citing a valid source ID does not by itself prove factual support.

| Key term | Meaning |
| --- | --- |
| **Context window** | The maximum amount of text a model can read in one call. |
| **Embedding** | A vector of numbers that represents the meaning of a text. |
| **Vector database** | A store that finds the vectors nearest to a query vector. |
| **Grounded answer** | An answer supported by, and citing, the retrieved sources. |

## Pattern · Retrieval-Augmented Generation

*Single-agent pattern from the [Architectural Patterns for Agentic AI](https://karthikv1392.github.io/agentic-patterns/) field guide.*

| | |
| --- | --- |
| **Problem** | A model needs relevant private or changing information to answer reliably. |
| **Solution** | Retrieve useful chunks, add them to the question, and ask the model to cite them. |
| **Flow** | Question → Retrieve → Add context → Grounded answer |
| **Trade-off** | Shorter prompts may reduce input usage, but retrieval and indexing have costs and can miss relevant evidence. |
| **In this lab** | Each microservices policy is one chunk. MiniLM embeds it, Chroma retrieves candidates, and we inspect the source IDs. |

## Main exercise at a glance

1. Create categories and policies
2. Ask without RAG: send the whole library each time
3. Embed and store the policies
4. Retrieve for several questions
5. Ask again using only the retrieved policies

## The two paths at a glance

```text
WITHOUT RAG
Question + all nine policies ──> Model ──> Answer or insufficient evidence

WITH RAG
Store the nine policies in a searchable collection
Question ──> Search ──> Two policies
Question + those two policies ──> Model ──> Answer or insufficient evidence
```

A **chunk** is one piece of text available for search. Each policy in this lab is short enough to be its own chunk. Each model request gets its own question and policy text.

## Step 1 · Create categories and policies
Run the cell to see nine policies grouped into three categories. IDs such as `P1` let you find and check cited policies later. The questions are stored separately in `questions`.

Find the policy you would show a model asked whether Reporting can read the Order service’s database. Keep that choice in mind when you see what the search returns.

In [ ]:
policies = {
    "Data ownership": [
        ("P1", "Each service owns its private database. Other services use its API or published events."),
        ("P2", "API composition joins small results by ID from services; broad searches may be expensive."),
        ("P3", "CQRS supports searchable order history with a separate read model that can lag behind writes."),
    ],
    "Resilience": [
        ("P4", "Retry a transient failure only for an idempotent operation, with a limit and backoff."),
        ("P5", "Do not blindly retry a card charge whose outcome is unknown; reconcile it first."),
        ("P6", "A bulkhead isolates resource pools so one slow dependency cannot exhaust the others."),
    ],
    "Messaging": [
        ("P7", "A consumer deduplicates an eventId with its business effect in one local transaction."),
        ("P8", "A transactional outbox stores the business change and publication intent in one local transaction."),
        ("P9", "A saga coordinates local transactions and compensates completed actions when needed."),
    ],
}
questions = ["May Reporting read the private database owned by the Order service?",
    "Support needs searchable order history by customer name across services. Which pattern fits?",
    "A payment charge timed out and may have succeeded. Should we retry it blindly?",
    "How should a consumer handle an event delivered more than once?",
    "What should we eat for lunch?"]
for category, items in policies.items():
    print(f"\n{category}:")
    for policy_id, text in items:
        print(f"  [{policy_id}] {text}")

## Step 2 · Ask without RAG: send the whole library each time
Start with a **baseline**, a result to compare against. The code sends all nine policies for each of the first two architecture questions and the final lunch question. It asks for a short answer with policy IDs such as `[P1]`, or the sentence “The supplied policies do not answer this question.” It saves the replies for Step 5.

In the output, find the full prompt, the model’s answer, and two ways to describe input size. A **word count** counts words in the user prompt. **Input tokens** count the text units the provider processed and can include the system message. Some providers leave that count unreported.

Sending extra text uses more input capacity and can increase cost. Finding a relevant detail can also get harder in very long prompts, as illustrated by the [Lost in the Middle study](https://arxiv.org/abs/2307.03172). This small library lets you observe what is sent in each method.

In [ ]:
records, lines = [], []
for category, items in policies.items():
    for policy_id, text in items:
        records.append((policy_id, category, text))
        lines.append(f"[{policy_id}] {category}: {text}")
all_policies = "\n".join(lines)
no_answer = "The supplied policies do not answer this question."
instructions = ("Use only these policies. Answer briefly and cite relevant IDs in square brackets. "
                f"If the policies cannot answer, reply only: {no_answer}")
comparison_questions = questions[:2] + questions[-1:]
baseline_words, baseline_tokens, baseline_answers = [], [], []
for question in comparison_questions:
    prompt = f"Question: {question}\nPolicies:\n{all_policies}\n{instructions}"
    baseline_words.append(len(prompt.split()))
    print("\n--- WITHOUT RAG: full prompt ---\n" + prompt)
    print("Prompt size:", len(prompt.split()), "words (not tokens)")
    answer = client.ask(prompt)
    tokens = client.trace[-1]["input_tokens"]
    baseline_tokens.append(tokens)
    baseline_answers.append(answer)
    print("Provider input tokens:", tokens if tokens is not None else "not reported")
    print("Model answer:", answer)

## Step 3 · Embed and store the policies
An **embedding** is a list of numbers that represents text. We use the same embedding model for a policy and a question so we can compare the two lists. Text about similar ideas often has similar embeddings even when it uses different words.

The first two output lines show the length and first five values of one policy embedding and one question embedding. Focus on the matching lengths. This cell recreates the lab’s collection from the current policies. Chroma stores each policy’s embedding, original text, ID, and category. Expect `Stored policy chunks: 9` with the original library.

In [ ]:
import chromadb
from chromadb.config import Settings
from chromadb.utils.embedding_functions import DefaultEmbeddingFunction

embed = DefaultEmbeddingFunction()
policy_vector, question_vector = embed([records[0][2], questions[0]])
print("Policy vector size:", len(policy_vector), "| first 5:", policy_vector[:5])
print("Question vector size:", len(question_vector), "| first 5:", question_vector[:5])
ids, documents, metadata = [], [], []
for policy_id, category, text in records:
    ids.append(policy_id)
    documents.append(text)
    metadata.append({"category": category})
db = chromadb.Client(Settings(anonymized_telemetry=False))
collection_name = "architecture-policies"
if collection_name in [item.name for item in db.list_collections()]:
    db.delete_collection(collection_name)
collection = db.create_collection(collection_name, embedding_function=embed,
                                 configuration={"hnsw": {"space": "cosine"}})
collection.add(ids=ids, documents=documents, metadatas=metadata)
print("Stored policy chunks:", collection.count())

## Step 4 · Retrieve for several questions
The search program, or **retriever**, turns each question into an embedding and finds the two policy embeddings closest to it. A lower **cosine distance** indicates a closer match by this measure. The cell prints each of the five questions with its two candidates.

Start with the database question. Is policy P1 among the two candidates? Then look at the unrelated lunch question. The search still returns two candidates because it ranks every question against the policies it has. Read the candidate text before treating it as relevant. This cell searches locally without asking the answer model.

In [ ]:
for question in questions:
    print("\nQUESTION:", question)
    result = collection.query(query_texts=[question], n_results=2)
    for policy_id, text, metadata, distance in zip(
            result["ids"][0], result["documents"][0], result["metadatas"][0], result["distances"][0]):
        print(f"  [{policy_id}] {metadata['category']} | distance {distance:.3f} | {text}")

## Step 5 · Ask again using only the retrieved policies
Run Steps 2 and 3 first to create the baseline and populate the searchable collection. The cell sends the same three questions from Step 2 with the same answer instructions. Here only the two policies returned by search go into each prompt.

For each question, compare the displayed word and input token counts, then read the two answers side by side. When input tokens are unavailable, the value is shown as `None`. Prompt size alone leaves out other costs such as searching and generating the answer.

Finally, check the citations. `Citation IDs valid: True` means the reply cites at least one ID in square brackets and the cited IDs came from retrieval. Read the policy text to decide whether it supports the claim. For lunch, expect the model to report insufficient evidence. In that case the citation check is marked `not needed`. For the first question, P1 says that other services use the owner’s API or published events.

In [ ]:
import re

for question, without_rag_words, without_rag_tokens, baseline_answer in zip(
        comparison_questions, baseline_words, baseline_tokens, baseline_answers):
    result = collection.query(query_texts=[question], n_results=2)
    context_lines = []
    for policy_id, text, metadata in zip(result["ids"][0], result["documents"][0], result["metadatas"][0]):
        context_lines.append(f"[{policy_id}] {metadata['category']}: {text}")
    context = "\n".join(context_lines)
    prompt = f"Question: {question}\nPolicies:\n{context}\n{instructions}"
    print("\n--- WITH RAG: retrieved prompt ---\n" + prompt)
    print("Retrieved IDs:", result["ids"][0])
    print("Prompt words: all policies =", without_rag_words, "| retrieved =", len(prompt.split()))
    answer = client.ask(prompt)
    tokens = client.trace[-1]["input_tokens"]
    print("Provider input tokens: all policies =", without_rag_tokens, "| retrieved =", tokens)
    cited = set(re.findall(r"\[(P\d+)\]", answer))
    retrieved = set(result["ids"][0])
    abstained = answer.strip() == no_answer
    print("Answer without RAG (Step 2):", baseline_answer)
    print("Answer with RAG:", answer)
    print("Reported insufficient evidence:", abstained)
    print("Citation IDs valid:", "not needed" if abstained else bool(cited) and cited.issubset(retrieved))

## What to look for

- Step 2 prints each baseline request with nine policies. Step 5 uses two for the same question.
- Step 4 can return results even for a question the library cannot answer.
- Step 5 asks about lunch too. If either answer invents a recommendation, point to the policies and explain why they provide no evidence for it.

## Try it
1. Pick the policy you expect for a timed-out card charge or duplicate event. Is it among the two candidates in Step 4?
2. Change `n_results` from 2 to 3 in Steps 4 and 5. Does the model get new supporting evidence?
3. Search for `eventId`. Compare the results with the policies that contain that exact term.
4. Add a policy about mobile BFFs. Run Steps 1–5 again and look for it among the search results. Try a new question in the first two positions to see its answer.
5. Move a relevant policy in the library and repeat the baseline. What else would you need to keep fixed to test whether its position matters?

## Takeaway
RAG chooses a subset of evidence before each answer. A shorter prompt can still work when the search finds the right evidence. Evaluate what was retrieved as well as what the model said.

## Optional reading · Other search techniques

| Technique | When you might use it |
| --- | --- |
| **Chunking** | Split long documents into smaller passages so a search can return the relevant portion. |
| **Semantic search** | Find text with related meaning, even when it uses different words. This lab uses embeddings to do it. |
| **Keyword search** | Look for exact words or identifiers such as `eventId`. |
| **Hybrid search** | Combine keyword matches and matches based on meaning. |
| **Metadata filters** | Search only material that matches a field such as category or version. |
| **Reranking** | Score the search results again before choosing what to send. |
| **Query rewriting** | Rephrase a short or ambiguous question before searching. |

See the [original RAG paper](https://arxiv.org/abs/2005.11401) and [Google's hybrid-search explanation](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/vector-search/about-hybrid-search).